# Modul 06 · Notebook 01 — Optimasi Inferensi (Quantization)

Di **nb00** kita lihat FP16 memakai **separuh** memori FP32. Sekarang kita melangkah lebih jauh: **quantization 4-bit**, yang memangkas memori jauh lebih banyak lagi — sehingga model besar bisa muat di GPU kecil seperti T4.

**Tujuan:** muat model yang **sama** (Qwen2.5-1.5B) dalam 4-bit, ukur penghematan memori vs FP16, dan pastikan jawabannya tetap masuk akal.

In [1]:
# Instal dependensi (pinned)
!pip -q install "transformers>=4.53,<5" "accelerate>=0.30" "bitsandbytes>=0.43"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s eta 0:00:00


## Apa itu quantization?

Bobot model aslinya disimpan sebagai angka presisi tinggi (FP32/FP16). **Quantization** menyimpannya dengan presisi lebih rendah — misalnya **4-bit** — seperti membulatkan harga ke ribuan terdekat: ukuran file jauh lebih kecil, ketelitian sedikit berkurang, tapi makna utamanya tetap.

- **NF4** (*4-bit NormalFloat*): format 4-bit yang dirancang khusus untuk bobot neural network.
- **compute_dtype**: walau bobot disimpan 4-bit, perhitungannya tetap dilakukan di **FP16**. Di T4 kita pakai **fp16 (bukan bf16)** — `bf16` baru optimal di GPU Ampere ke atas dan bisa bermasalah dengan 4-bit di T4.

In [2]:
# Helper: muat model dengan konfigurasi tertentu, ukur memori bobot
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)

def load_mem(quant=None, dtype=None):
    torch.cuda.empty_cache(); gc.collect()
    kw = {"device_map": "auto"}
    if quant is not None: kw["quantization_config"] = quant
    if dtype is not None: kw["dtype"] = dtype
    model = AutoModelForCausalLM.from_pretrained(NAME, **kw)
    return model, torch.cuda.memory_allocated() / 1e9

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
# 1) FP16 sebagai pembanding (lalu dibebaskan)
m16, mem16 = load_mem(dtype=torch.float16)
del m16; gc.collect(); torch.cuda.empty_cache()

# 2) 4-bit (NF4, compute fp16) -- model ini kita simpan untuk dipakai generate
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16 (BUKAN bf16) -> aman di T4
    bnb_4bit_use_double_quant=True,
)
m4, mem4 = load_mem(quant=bnb)

print(f"Memori model FP16  : {mem16:.2f} GB")
print(f"Memori model 4-bit : {mem4:.2f} GB   -> {mem16/mem4:.1f}x lebih hemat dari FP16")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Memori model FP16  : 3.09 GB
Memori model 4-bit : 1.16 GB   -> 2.7x lebih hemat dari FP16


In [4]:
# Apakah jawaban model 4-bit tetap masuk akal?
ids = tok.apply_chat_template(
    [{"role": "user", "content": "Sebutkan 3 manfaat quantization untuk menjalankan LLM."}],
    add_generation_prompt=True, return_tensors="pt").to(m4.device)
out = m4.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=120,
                  do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Quantization adalah proses mengubah representasi data dari satu tingkat ke tingkat lainnya. Dalam konteks LLM (Large Language Model), beberapa manfaat utama yang dapat dihasilkan melibatkan:

1. Efisiensi dalam memori dan waktu: Quantization dapat membantu menurunkan ukuran model secara signifikan, mengurangi jumlah bit yang diperlukan untuk menyimpan dan menganalisai informasi. Ini bisa memberikan efisiensi lebih baik dalam hal memori dan waktu.

2. Kompresi dalam peng


## Tangga memori: FP32 → FP16 → 4-bit

| Presisi | Memori bobot (Qwen2.5-1.5B) | Catatan |
|---------|------------------------------|---------|
| FP32    | ~6.2 GB                      | presisi penuh (dari nb00) |
| FP16    | ~3.1 GB                      | separuh, kualitas ≈ sama (dari nb00) |
| 4-bit   | ~1.0–1.3 GB                  | hemat besar, sedikit turun ketelitian |

**Kapan pakai apa?**
- **FP16** — default aman; kualitas penuh, memori separuh.
- **4-bit** — saat model **tidak muat** di GPU (mis. model 7B di T4) atau ingin melayani lebih banyak request. Trade-off: sedikit penurunan kualitas.

## Jalur lain: meng-*compile* model (TensorRT)

Selain memperkecil bobot, NVIDIA punya **TensorRT** — *compiler* yang mengubah model menjadi *engine* yang sangat dioptimalkan untuk GPU tertentu (fuse kernel, pilih presisi, dll). Alurnya: `model → ONNX → TensorRT engine`.

> ⚠️ **Resep yang terbukti (untuk eksplorasi mandiri):** `tensorrt<11`, `tf2onnx>=1.17`, dan jalur **TF-TRT sudah mati di TF 2.18+** → gunakan **ONNX → TRT**. Build engine cukup berat & rapuh di Colab, jadi di kursus ini TensorRT kita **kenalkan secara konsep**, bukan build penuh. Untuk produksi nyata, NVIDIA NIM (nb02) sudah membungkus optimasi ini untukmu.

## 🧪 Latihan

1. Ubah `bnb_4bit_use_double_quant` menjadi `False` — apakah memori 4-bit berubah?
2. Bandingkan **panjang jawaban** & kualitas antara model FP16 (nb00) dan 4-bit di sini untuk prompt yang sama. Terasa bedanya?
3. (Lanjutan) Coba muat model **lebih besar** (mis. `Qwen/Qwen2.5-3B-Instruct`) dalam 4-bit — apakah muat di T4?